# Library installations (May require to Restart session from runtime after installation or sometimes after import)









In [1]:
%%capture --no-display

!apt-get install -y coinor-cbc

!pip install cylp
!pip install pygamma-agreement
!pip install optlang cylp
!pip install matplotlib

# Part.0 - Libraries imports and common funtions.

In [2]:
#Libraries import.
from pathlib import * #Not used
from collections import Counter
from datetime import datetime
import pytz
import os # Define the files to be deleted and modified in google Colab
import csv #Save to CSV after calculations
import zipfile #For zip files
import numpy as np
from collections import defaultdict

import xml.etree.ElementTree as ET
import html
from logging import INFO

import re

from google.colab import files
import glob
import json
import shutil
import tempfile
import sys

import string
from pytz import timezone
import matplotlib.pyplot as plt
from pyannote.core import Segment
from pygamma_agreement import (CombinedCategoricalDissimilarity, Continuum)
from pygamma_agreement import show_continuum

import logging
import pandas as pd

In [ ]:
# Last run date and time

# Get current time in Berlin
tz_Berlin = pytz.timezone('Europe/Berlin')
current_date_timex = datetime.now(tz_Berlin).strftime('%d.%m.%Y %HH:%MM')

print("Last run date and time:", current_date_timex)

Last run date and time: 06.07.2025 03H:32M


### Common functions

In [ ]:
def action_delete_all_files_list(paths):
    '''
    Deletes files or directories listed in `paths`.
    '''
    try:
        for path in paths:
            if os.path.isfile(path) or os.path.islink(path):
                os.remove(path)
                print(f"☠️ Deleted file: {path}")
            elif os.path.isdir(path):
                shutil.rmtree(path)
                print(f"☠️ Deleted directory: {path}")
            else:
                print(f"⚠️ Path does not exist or is not a file/directory: {path}")
    except Exception as e:
        print(f"⚠️ Error while deleting files: {e}")

In [5]:
def action_delete_all_files(directory):
  '''
  Delete files and folders.
  '''
  try:
      # Check if the directory exists
      if os.path.exists(directory) and os.path.isdir(directory):
          # Iterate over all files and directories within the directory
          for filename in os.listdir(directory):
              file_path = os.path.join(directory, filename)

              # Check if it's a file and delete it
              if os.path.isfile(file_path) or os.path.islink(file_path):
                  os.remove(file_path)
                  print(f"☠️ Deleted file: {file_path}")
              # Check if it's a directory (optional) and skip or delete recursively
              elif os.path.isdir(file_path):
                  shutil.rmtree(file_path)  # Use this if you want to delete subdirectories
                  print(f"☠️ Deleted directory: {file_path}")
      else:
          print("⚠️ Directory does not exist or is not a directory.")
  except Exception as e:
      print(f"⚠️ Error while deleting files: {e}")

In [6]:
# Path to the folder in Google Drive where the files are stored
drive_folder_path = '/content/'

# delete all files
action_delete_all_files(drive_folder_path)

☠️ Deleted directory: /content/.config
☠️ Deleted directory: /content/sample_data


# Part.A - Preprocessing

## 1- To convert the inception(annotation) json to csv
```
#Input:
evaluative-structures-and-cultural-cri-1-webanno.custom.Wertung.json
evaluative-structures-and-cultural-cri-1-webanno.custom.AltNeuCodierung.json

#output
Files:(optional)
"0_inception_annotators_Wertung.csv"
"0_inception_annotators_AltNeuCodierung.csv"

Folders:
"0_inception_annotators_Wertung"
"0_inception_annotators_AltNeuCodierung"
```

### Parameters

In [7]:
# Replace with your actual JSON file name

json_files = ["evaluative-structures-and-cultural-cri-1-webanno.custom.Wertung.json",
              "evaluative-structures-and-cultural-cri-1-webanno.custom.AltNeuCodierung.json"]

In [ ]:
# Folder to check
folder_path = r"/content/"

# Step 1: Check if the folder exists
if not os.path.isdir(folder_path):
    raise FileNotFoundError(f"🚫 Folder not found: {folder_path}\n⛔ Stopping execution.")

# Step 2: Check if the folder contains the required JSON files
file_1 = os.path.join(folder_path, json_files[0])
file_2 = os.path.join(folder_path, json_files[1])

if not os.path.isfile(file_1):
    raise FileNotFoundError(f"🚫 File not found: {file_1}\n⛔ Stopping execution.")

if not os.path.isfile(file_2):
    raise FileNotFoundError(f"🚫 File not found: {file_2}\n⛔ Stopping execution.")

print(f"✅ Folder exists and contains both files: \n\n{file_1} \n\nand\n\n{file_2} \n\nReady to go! 🟢")


✅ Folder exists and contains both files: 

/content/evaluative-structures-and-cultural-cri-1-webanno.custom.Wertung.json 

and

/content/evaluative-structures-and-cultural-cri-1-webanno.custom.AltNeuCodierung.json 

Ready to go! 🟢


In [ ]:
'''
Optional: Not required
Converts inception json to single csv for all annotators with additional column of evaluation having constant value of 1.
'''
for json_file in json_files:
  # Read JSON data
  with open(json_file, "r", encoding="utf-8") as f:
      data = json.load(f)

  # Remove '.txt' from the 'doc' field
  for item in data:
      item["Filename"] = os.path.splitext(item.pop("doc"))[0]  # Rename "doc" to "Filename"
      item["start"] = item.pop("begin")  # Rename "begin" to "start"
      item["end"] = item.pop("end")      # No change needed if already correct
      item["text_input"] = item.pop("text")  # Rename "text" to "text_input"

      item["evaluation"] = "1" #Add additional column for pygamma.

  # Write to CSV
  if "AltNeuCodierung.json" in json_file:
    csv_file = "0_inception_annotators_AltNeuCodierung.csv"
  else:
    csv_file = "0_inception_annotators_Wertung.csv"

  with open(csv_file, mode="w", encoding="utf-8", newline="") as f:
      writer = csv.DictWriter(f, fieldnames=["Filename", "user", "start", "end", "text_input","evaluation"])
      writer.writeheader()
      writer.writerows(data)

  print(f"✅ Converted {json_file} to {csv_file} with updated column names and '.txt' removed from 'Filename in columns'\n")

✅ Converted evaluative-structures-and-cultural-cri-1-webanno.custom.Wertung.json to 0_inception_annotators_Wertung.csv with updated column names and '.txt' removed from 'Filename in columns'

✅ Converted evaluative-structures-and-cultural-cri-1-webanno.custom.AltNeuCodierung.json to 0_inception_annotators_AltNeuCodierung.csv with updated column names and '.txt' removed from 'Filename in columns'



In [ ]:
'''
Converts inception json to multiple csv's with filenames with additional column of evaluation having constant value of 1.
'''

for json_file in json_files:
  # Set output paths
  if "AltNeuCodierung.json" in json_file:
    output_base = "0_inception_annotators_AltNeuCodierung"
  else:
    output_base = "0_inception_annotators_Wertung"

  print(f"\n\n⌛️ Processing: {json_file}\n")
  # Read JSON data
  with open(json_file, "r", encoding="utf-8") as f:
      data = json.load(f)

  # Process and rename fields
  for item in data:
      item["Filename"] = os.path.splitext(item.pop("doc"))[0]  # Remove '.txt'
      item["start"] = item.pop("begin")
      item["end"] = item.pop("end")
      item["text_input"] = item.pop("text")

      item["evaluation"] = "1"  # Add additional column for pygamma

  # Group by Filename (no longer by user)
  grouped_data = {}
  for item in data:
      key = item["Filename"]
      if key not in grouped_data:
          grouped_data[key] = []
      grouped_data[key].append(item)

  # Save CSVs for each Filename group
  for filename, records in grouped_data.items():
      folder_path = os.path.join(output_base, '')  # Folder based on Filename
      os.makedirs(folder_path, exist_ok=True)

      output_csv_path = os.path.join(folder_path, filename + ".csv")
      with open(output_csv_path, mode="w", encoding="utf-8", newline="") as f:
          writer = csv.DictWriter(f, fieldnames=["Filename", "user", "start", "end", "text_input", "evaluation"])
          writer.writeheader()
          writer.writerows(records)

      print(f"✅ CSaved {len(records)} records to {output_csv_path}")




⌛️ Processing: evaluative-structures-and-cultural-cri-1-webanno.custom.Wertung.json

✅ CSaved 26 records to 0_inception_annotators_Wertung/Aichinger_Das_Fenstertheater.csv
✅ CSaved 14 records to 0_inception_annotators_Wertung/Altenberg_Die_Natur.csv


⌛️ Processing: evaluative-structures-and-cultural-cri-1-webanno.custom.AltNeuCodierung.json

✅ CSaved 7 records to 0_inception_annotators_AltNeuCodierung/Aichinger_Das_Fenstertheater.csv
✅ CSaved 16 records to 0_inception_annotators_AltNeuCodierung/Altenberg_Die_Natur.csv


# Part.B Using inception(json) and Pygamma

## 1- Preparing data of inception folders for pygamma

```
input(folder and files within to colab): 0_txt_files_original/0_txt_files_cleaned +  [ 0_gold_standard_xmi ] + evaluative-structures-and-cultural-criti-webanno.custom.Wertung.json

output(folder): annotators.csv + 6_pygamma_csv_annotator1, 6_pygamma_csv_annotator2, 7_pygamma_output_[*]

```

In [ ]:

# Path to the folder in Google Drive where the files are stored
base_paths = [
    "0_inception_annotators_AltNeuCodierung.csv",
    "0_inception_annotators_Wertung.csv"
]

prefix = "/content/"
files_folders_to_delete = [prefix + path for path in base_paths]

# delete all files
action_delete_all_files_list(files_folders_to_delete)

⚠️ Path does not exist or is not a file/directory: /content/0_inception_annotators_AltNeuCodierung.csv
⚠️ Path does not exist or is not a file/directory: /content/0_inception_annotators_Wertung.csv


In [ ]:
annotator_csv_dir_list = ["0_inception_annotators_AltNeuCodierung","0_inception_annotators_Wertung"]

existing_annotator_folders = [folder for folder in annotator_csv_dir_list if os.path.exists(folder)]

print("Existing folders:", existing_annotator_folders)

Existing folders: ['0_inception_annotators_AltNeuCodierung', '0_inception_annotators_Wertung']


In [ ]:
# Check if the folder exists
if len(existing_annotator_folders)<2:
    raise FileNotFoundError(f"🚫 Check for folders, min 2 required: {existing_annotator_folders}\n⛔ Stopping execution.")

# Check if folder contains at least one .csv file
for folder_path in existing_annotator_folders:
  csv_files = glob.glob(os.path.join(folder_path, '*.csv'))
  if not csv_files:
      raise FileNotFoundError(f"🚫 No CSV files found in folder: {folder_path}\n⛔ Stopping execution.")

print(f'✅ Folder "{existing_annotator_folders}" exists and contains csv file(s). Ready to go! 🟢')


✅ Folder "['0_inception_annotators_AltNeuCodierung', '0_inception_annotators_Wertung']" exists and contains csv file(s). Ready to go! 🟢


## 2- using pygamma on annotations.

In [ ]:
# Annotators and Paths to folders
base_paths = ["0_inception_annotators_Wertung","0_inception_annotators_AltNeuCodierung"]

prefix = "/content/"
annotated_folders_input = [prefix + path for path in base_paths]

existing_annotator_folders_input = [folder for folder in annotated_folders_input if os.path.exists(folder)]

print("Existing folders:", existing_annotator_folders_input)


Existing folders: ['/content/0_inception_annotators_Wertung', '/content/0_inception_annotators_AltNeuCodierung']


In [ ]:
'''
Incase of Inception need one folder: "0_inception_annotators"
OR annotated_folders_input(list of folders)
existing_annotator_folders_input
'''

# Load CSV file
def load_csv(file_path):
    return pd.read_csv(file_path)

# Compute agreement for each file
def compute_agreement_inception_files(filename, file_path1, file_path2 = '' ):
    df1 = load_csv(file_path1)
    #df1 = df1[~df1.apply(lambda row: row.astype(str).str.startswith('-')).any(axis=1)]
    #df2 = load_csv(file_path2)

    continuum = Continuum()
    annotator_x = []
    for _, row in df1.iterrows():
        if(row['evaluation'] == 'NEUTRAL'):
          continue
        if ( row['start'] =='-'):
          continue
        if( row['user'] not in annotator_x):
          annotator_x.append(row['user'])

        continuum.add(row['user'], Segment(int(row['start']), int(row['end'])), row['evaluation'])

        dissim = CombinedCategoricalDissimilarity(alpha=1, beta=2)
    gamma_results = continuum.compute_gamma(dissim)

    # Show and save continuum plot
    show_continuum(continuum, labelled=False)
    result = gamma_results.gamma

    # Modify filename for title
    clean_filename = filename.replace(".csv", "")  # Remove suffix

    plt.xlabel("Token Position")
    plt.title(f"{clean_filename} - Agreement Score: {result:.6f}")

    # Save plot
    plot_path = os.path.join(output_folder, f"{clean_filename}.png")
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()  # Close plot to free memory

    return clean_filename, annotator_x, result

# Set logging level to ERROR to suppress warnings
logging.getLogger().setLevel(logging.ERROR)

#Modify for more runs.
total_runs_required = 1
for i in range(total_runs_required): # Change if want to do multiple runs of pygamma
  print('\nrun number:',i,"\n")

  _run_number = 0
  for folder_item in existing_annotator_folders_input:
    print(f"\n============ Processing files in: {folder_item}============\n")
    folder1 = folder_item
    #folder2 = ''  # Uncomment and set if you have a second folder to compare
    
    folder_name = os.path.basename(folder1)

    output_folder = "/content/1_pygamma_output/"+folder_name # Output folder for plots & CSV #TODO change to appropriate name.

    # Ensure output folder exists
    os.makedirs(output_folder, exist_ok=True)
    _run_number = 0

    file_path = os.path.join(output_folder, "agreement_scores.csv")
    _run_number = 0  # Default value

    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        if "consecutive_run_number_pygamma" in df.columns:
            try:
                _run_number = int(df["consecutive_run_number_pygamma"].max()) + 1
            except (ValueError, TypeError):
                _run_number = 0
    else:
        _run_number = 0


    # Main execution loop
    if __name__ == "__main__":
        # List all files in folder1 (assuming both folders have the same filenames)
        files1 = {f for f in os.listdir(folder1) if not f.startswith('.')}
        
        # If you have a second folder, uncomment the next lines and modify accordingly
        #files2 = {f for f in os.listdir(folder2) if not f.startswith('.')}

        # Find common files in both folders
        #common_files = files1.intersection(files2)
        common_files = files1

        # Store results for CSV
        results = []
        _index = 0

        for filename in sorted(common_files):  # Sort to maintain order

            _index += 1
            print(f"\n{_index:02} =========== Processing: {filename} ====================")

            file_path1 = os.path.join(folder1, filename)
            print(file_path1)

            df1 = pd.read_csv(file_path1, index_col=False)
            if df1["user"].nunique() < 2:
                print(f"{_index:02} ❌ Skipping: {filename}, fewer than 2 unique annotators\n")
                continue

            # If you have a second folder, uncomment the next line and modify accordingly
            file_path2 = '' #os.path.join(folder2, filename)

            print(f"{_index:02} Processing {filename}...")
            try:
              clean_filename, annotator_x,agreement_score = compute_agreement_inception_files(filename, file_path1, file_path2)
            except ValueError:
              print("❌ File have less than 2 annotators, skipping: ",filename)
              continue
            # Append results to list
            if "inception" in folder1:
              _prompt = "-"
              _role = "-"
              _remarks = f"pygamma run: {current_date_timex}"

            else:
              _prompt = df1["prompt_x"][1]
              _role = df1["role"][1]
              _remarks = df1["remarks"][1]

            results.append([clean_filename, annotator_x, agreement_score,_prompt,_role,_remarks])

            print(f"✅ {filename} - {annotator_x} - Inter-Annotator Agreement Score: {agreement_score:.6f}\n")

        # Save results as CSV
        results_df = pd.DataFrame(results, columns=["filename", "annotators", "agreement_score","prompt","role","remarks"])
        results_df['consecutive_run_number_pygamma'] = _run_number
        results_csv_path = os.path.join(output_folder,"agreement_scores.csv")

        # Check if file exists
        file_exists = os.path.isfile(results_csv_path)
        if file_exists:
            # Append to CSV without header if file exists
            results_df.to_csv(results_csv_path, mode='a', header=not file_exists, index=False)
        else:
            # Create new CSV with header
            results_df.to_csv(results_csv_path, index=False)

        print(f"✅ 🎉 ============== Results saved in: {results_csv_path} ============\n")

# Set logging level to WARNING to suppress INFO messages
logging.getLogger().setLevel(logging.WARNING)


run number: 0 


============ Processing files in: /content/0_inception_annotators_Wertung============


01 =========== Processing: Aichinger_Das_Fenstertheater.csv ====================
/content/0_inception_annotators_Wertung/Aichinger_Das_Fenstertheater.csv
01 Processing Aichinger_Das_Fenstertheater.csv...
✅ Aichinger_Das_Fenstertheater.csv - ['Fiona', 'Melanie'] - Inter-Annotator Agreement Score: 0.325800


02 =========== Processing: Altenberg_Die_Natur.csv ====================
/content/0_inception_annotators_Wertung/Altenberg_Die_Natur.csv
02 Processing Altenberg_Die_Natur.csv...
✅ Altenberg_Die_Natur.csv - ['Fiona', 'Melanie'] - Inter-Annotator Agreement Score: 0.559684

✅ 🎉 ============== Results saved in: /content/1_pygamma_output/0_inception_annotators_Wertung/agreement_scores.csv ============ 

============ Processing files in: /content/0_inception_annotators_AltNeuCodierung============


01 =========== Processing: Aichinger_Das_Fenstertheater.csv ====================
/content

In [ ]:
#Statistics

# Paths to the CSV files for agreement scores
agreement_scores_csv_paths = ["/content/1_pygamma_output/0_inception_annotators_AltNeuCodierung/",
            "/content/1_pygamma_output/0_inception_annotators_Wertung/"]


for agreement_score_path in agreement_scores_csv_paths:
  # Load CSV file
  agreement_score_csv = agreement_score_path + "agreement_scores.csv"
  df = pd.read_csv(agreement_score_csv)

  # Group by 'filename' and calculate mean, median, standard deviation, and variance of 'agreement_score'
  grouped_stats = df.groupby("filename")["agreement_score"].agg(['mean', 'median', 'std', 'var'])

  # Get the longest 'annotators' string per filename
  longest_annotators = df.groupby("filename")["annotators"].apply(lambda x: max(x, key=len)).reset_index()

  # Merge the two DataFrames on filename
  merged_df = pd.merge(grouped_stats, longest_annotators, on="filename")

  # Print and save the result
  merged_df.to_csv(f"{agreement_score_path}/pygamma_statistics.csv", index=False)


In [21]:
# Get current time in Berlin
tz_Berlin = pytz.timezone('Europe/Berlin')
current_date_timex = datetime.now(tz_Berlin).strftime('%d-%m-%Y_%HH%MM')

!pip freeze > requirements_{current_date_timex}.txt

# Part.C - Save all required folders and create a zip

### Create zip

In [ ]:
# Get current time in Berlin
tz_Berlin = pytz.timezone('Europe/Berlin')
current_timex = datetime.now(tz_Berlin).strftime('%d-%m-%Y')

# Define the folders to zip
folders_to_zip = ["/content/"]

# Function to ignore .ipynb_checkpoints
def ignore_checkpoints(dir, files):
    return [f for f in files if f == '.ipynb_checkpoints']

if len(folders_to_zip) > 1:
    # Create a temp directory to hold copies
    with tempfile.TemporaryDirectory() as temp_dir:
        for folder in folders_to_zip:
            folder = folder.rstrip('/')
            folder_name = os.path.basename(folder)
            dest_path = os.path.join(temp_dir, folder_name)
            shutil.copytree(folder, dest_path, ignore=ignore_checkpoints)

        output_filename = f"/content/annotation_agreement_pygamma_results_selective_{current_timex}"

        shutil.make_archive(output_filename, 'zip', root_dir=temp_dir)
        print(f"Zipped multiple folders into {output_filename}.zip")

else:
    for folder in folders_to_zip:
        folder = folder.rstrip('/')
        folder_name = os.path.basename(folder)
        parent_dir = os.path.dirname(folder) or '.'
        output_filename = f"/content/annotation_agreement_pygamma_results_Full_{current_timex}"

        shutil.make_archive(output_filename, 'zip', root_dir=parent_dir, base_dir=folder_name)
        print(f"✅ Zipped {folder} as {output_filename}.zip")

✅ Zipped /content as /content/annotation_agreement_pygamma_results_Full_06-07-2025.zip
